In [3]:
import pandas as pd

##### read the data from each data set 
{
    egypt local prand,
    fragrantica
 }

In [144]:
path = "/mnt/d/Projects/perfume-market-analysis/data/processed/"
df_egy = pd.read_csv(path + "egypt/egypt_fragrance.csv")
df_frag = pd.read_csv(path + "fragrantica/fra_cleaned.csv", encoding="cp1252", sep=";", engine="python")
df_frag['Rating Value'] = df_frag['Rating Value'].str.replace(',', '.').astype(float)
df_frag = df_frag[df_frag['Rating Value'] >= 3.79]
df_frag = df_frag[df_frag['Rating Count'] >= 160]
df_frag = df_frag[df_frag['Gender'] != 'women']
df_frag = df_frag[df_frag['Perfume'].str.len() != 1]
df_frag = df_frag[df_frag['Perfume'].str.len() != 2]
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'intense']
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'uomo']
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'homme']
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'l-homme']
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'oud']
df_frag = df_frag[df_frag['Perfume'].str.lower() != 'noir']

### time to play: 
- the main target is to concat them and get the main top notes into mine data so complete my data
- the problem is how to combine them under one umbrella??

In [145]:
df_frag.info()

<class 'pandas.DataFrame'>
Index: 4355 entries, 5944 to 24039
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   url           4355 non-null   str    
 1   Perfume       4355 non-null   str    
 2   Brand         4355 non-null   str    
 3   Country       4355 non-null   str    
 4   Gender        4355 non-null   str    
 5   Rating Value  4355 non-null   float64
 6   Rating Count  4355 non-null   int64  
 7   Year          4172 non-null   float64
 8   Top           4355 non-null   str    
 9   Middle        4355 non-null   str    
 10  Base          4355 non-null   str    
 11  Perfumer1     4355 non-null   str    
 12  Perfumer2     291 non-null    str    
 13  mainaccord1   4355 non-null   str    
 14  mainaccord2   4354 non-null   str    
 15  mainaccord3   4340 non-null   str    
 16  mainaccord4   4302 non-null   str    
 17  mainaccord5   4224 non-null   str    
dtypes: float64(2), int64(1), str(15)
memory 

In [146]:
df_egy['inspired_by_key'] = df_egy['inspired_by'].str.replace(r'[^a-zA-Z0-9]+', "-",regex=True).str.lower()

In [147]:
df_merge = df_egy.merge(df_frag[['Perfume','Top','Middle', 'Base', 'Perfumer1', 'mainaccord1', 'mainaccord2', 'mainaccord3']],
                how='left',
                left_on='inspired_by_key',
                right_on='Perfume')

In [148]:
df_merge['Perfume'].value_counts()

Perfume
scandal-pour-homme                   14
y-eau-de-parfum                      12
le-male-elixir                       10
bleu-de-chanel                        9
ultra-male                            9
                                     ..
elysium-pour-homme-parfum-cologne     2
smoking-hot                           1
god-of-fire                           1
vibrato                               1
apple-brandy-on-the-rocks             1
Name: count, Length: 67, dtype: int64

> if there is match then rememper it in matched_type as exact

In [149]:
df_merge.loc[~df_merge['Top'].isna(), 'matched_type'] = 'exact'

In [ ]:
for i, row in df_merge[df_merge['matched_type'].isnull()].iterrows():
    matches_rows = df_frag[df_frag['Perfume'].apply(lambda x: x.lower() in row['inspired_by_key'].lower())]
    # print(matches_rows['Perfume'])
    if not matches_rows.empty:
        df_merge.loc[i, 'matched_type'] = 'partial'
        df_merge.loc[i, 'Perfume'] = matches_rows['Perfume'].values[0]
        df_merge.loc[i, 'Top'] = matches_rows['Top'].values[0]
        df_merge.loc[i, 'Middle'] = matches_rows['Middle'].values[0]
        df_merge.loc[i, 'Base'] = matches_rows['Base'].values[0]
        df_merge.loc[i, 'Perfumer1'] = matches_rows['Perfumer1'].values[0]
        df_merge.loc[i, 'mainaccord1'] = matches_rows['mainaccord1'].values[0]
        df_merge.loc[i, 'Brand'] = matches_rows['Brand'].values[0]
        df_merge.loc[i, 'mainaccord2'] = matches_rows['mainaccord2'].values[0]
        df_merge.loc[i, 'mainaccord3'] = matches_rows['mainaccord3'].values[0]

In [152]:
df_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   brand            1134 non-null   str    
 1   product_name     1134 non-null   str    
 2   inspired_by      1134 non-null   str    
 3   size_unit        1134 non-null   str    
 4   variant_type     1134 non-null   str    
 5   price            1134 non-null   float64
 6   available        1134 non-null   bool   
 7   identity         1134 non-null   str    
 8   product_url      1134 non-null   str    
 9   collected_date   1134 non-null   str    
 10  size             1134 non-null   float64
 11  price_per_unit   1134 non-null   float64
 12  top_notes        164 non-null    str    
 13  heart_notes      166 non-null    str    
 14  base_notes       166 non-null    str    
 15  review_count     140 non-null    float64
 16  brand_name       171 non-null    str    
 17  rate             30 non-n

In [154]:
df_merge.to_csv(path + "egypt/egypt_fragrance_merged.csv", index=False)